# **Purpose**

**Task:** context + answer → question

This notebook fine-tunes `TinyLlama/TinyLlama-1.1B-Chat-v1.0` using `LoRA/QLoRA` on a subset of SQuAD to generate a question given a context passage and an answer span.

**Not gated:** unlike the Llama 3.2 version, TinyLlama is a fully open, ungated checkpoint on the Hugging Face Hub, so no access request is required. 

## **Install dependencies**

In [1]:
!pip install -qU \
 transformers==5.16.1 \
 accelerate==1.14.0 \
 peft==0.18.1 \
 bitsandbytes==0.48.2 \
 datasets


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 57.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 75.3 MB/s eta 0:00:00


## **Imports**

In [2]:
import numpy as np
import torch
import transformers
import accelerate
import peft

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    DataCollatorForSeq2Seq,
    TrainingArguments,
    Trainer,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel

from kaggle_secrets import UserSecretsClient
import os

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("accelerate:", accelerate.__version__)
print("peft:", peft.__version__)


torch: 2.10.0+cu128
transformers: 5.16.1
accelerate: 1.14.0
peft: 0.18.1


## **Load the API Keys and Tokens**

TinyLlama is not gated, so `HF_TOKEN` isn't required to *load* the base model. Keep this cell if you plan to `push_to_hub` your trained adapter/model at the end; otherwise it's safe to skip.

In [3]:
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")   # must match the exact secret name you set in Kaggle Secrets, only needed for pushing to the Hub

os.environ["HF_TOKEN"] = hf_token


## **Config**

Tweak these for experimenting. `TRAIN_SUBSET_SIZE` / `VAL_SUBSET_SIZE` control how much of SQuAD you use - start small to sanity-check the pipeline before scaling up.

`USE_4BIT` toggles QLoRA (4-bit base model + LoRA adapters) vs. plain LoRA (16-bit base model + LoRA adapters). TinyLlama is small enough (1.1B params) that plain LoRA in fp16/bf16 is very feasible on a single T4/L4, but QLoRA is left on by default to keep memory headroom generous and to mirror the Llama 3.2 3B notebook.


In [4]:
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"   # open, ungated - no access request needed
MAX_INPUT_LENGTH = 512                            # context+answer prompt length (same budget as the Flan-T5 notebook)
MAX_TARGET_LENGTH = 96                            # questions are short
MAX_SEQ_LENGTH = MAX_INPUT_LENGTH + MAX_TARGET_LENGTH + 32   # causal LM sees prompt+completion in one sequence
TRAIN_SUBSET_SIZE = 10000                          # subset of SQuAD train split, set to None for full data
VAL_SUBSET_SIZE = 10
TRAIN_EPOCH_SIZE = 2
OUTPUT_DIR = "/content/tinyllama-1.1b-qg-lora"
SEED = 42

# --- PEFT / QLoRA config ---
USE_4BIT = True             # QLoRA (4-bit base model) if True, plain LoRA (16-bit base model) if False
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

SYSTEM_PROMPT = (
    "You are a question generation assistant. Given a context passage and a target answer, "
    "generate a single question whose correct answer is exactly the target answer."
)

# *****IMPORTANT*****
# Change the Huggingface pushing directory below


## **Load SQuAD and take a subset**

Uses the Hugging Face `squad` dataset. Swap to `"squad_v2"` if you also want unanswerable examples (note: `squad_v2` has empty answer lists for some examples, which the preprocessing below already handles gracefully).


In [5]:
raw = load_dataset("squad")

train_ds = raw["train"].shuffle(seed=SEED)
val_ds = raw["validation"].shuffle(seed=SEED)

if TRAIN_SUBSET_SIZE:
    train_ds = train_ds.select(range(TRAIN_SUBSET_SIZE))
if VAL_SUBSET_SIZE:
    val_ds = val_ds.select(range(VAL_SUBSET_SIZE))

print(train_ds)
print(val_ds)


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

plain_text/validation-00000-of-00001.par(…):   0%|          | 0.00/1.82M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

Dataset({
    features: ['id', 'title', 'context', 'question', 'answers'],
    num_rows: 10000
})
Dataset({
    features: ['id', 'title', 'context', 'question', 'answers'],
    num_rows: 10
})


In [6]:
# Peek at one example
train_ds[0]


{'id': '573173d8497a881900248f0c',
 'title': 'Egypt',
 'context': 'The Pew Forum on Religion & Public Life ranks Egypt as the fifth worst country in the world for religious freedom. The United States Commission on International Religious Freedom, a bipartisan independent agency of the US government, has placed Egypt on its watch list of countries that require close monitoring due to the nature and extent of violations of religious freedom engaged in or tolerated by the government. According to a 2010 Pew Global Attitudes survey, 84% of Egyptians polled supported the death penalty for those who leave Islam; 77% supported whippings and cutting off of hands for theft and robbery; and 82% support stoning a person who commits adultery.',
 'question': 'What percentage of Egyptians polled support death penalty for those leaving Islam?',
 'answers': {'text': ['84%'], 'answer_start': [468]}}

## **Load tokenizer and (quantized) model**

When `USE_4BIT` is on, the base model is loaded in 4-bit NF4 precision via `bitsandbytes` and prepared for k-bit training - this is the "QLoRA" part. LoRA adapters (added in the next section) are trained on top in full/half precision regardless.

TinyLlama-1.1B-Chat ships with its own chat template (a ChatML-style format), so `tokenizer.apply_chat_template` works out of the box just like it does for the Llama 3.2 Instruct tokenizer.


In [7]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=hf_token)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"   # right padding for training with causal LMs

compute_dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
) if USE_4BIT else None

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=compute_dtype,
    token=hf_token,
)

if USE_4BIT:
    model = prepare_model_for_kbit_training(model)

model.config.use_cache = False   # required alongside gradient checkpointing


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

## **Configure LoRA**

Wraps the base model with LoRA adapters on the attention and MLP projection layers. Only these adapter weights (a small fraction of total parameters) get trained and saved - the base model stays frozen. TinyLlama uses the same Llama-style architecture (q/k/v/o + gate/up/down projections), so the target module names carry over unchanged from the Llama 3.2 notebook.


In [8]:
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


trainable params: 12,615,680 || all params: 1,112,664,064 || trainable%: 1.1338


## **Preprocessing**

Builds the same "Target Answer / Context" prompt as the Flan-T5 notebook, but wraps it in TinyLlama's chat template (system + user turn) and appends the gold question as the assistant turn. Since this is a causal LM, prompt and completion are packed into a single sequence - the loss is masked (`-100`) over the prompt tokens so the model is only trained to predict the question itself.

**Tip:** for the "answer highlighting" trick used in a lot of QG literature, wrap the answer span inside the context with a marker (e.g. `<hl> {answer} <hl>`) before building the prompt - this can measurably improve which part of the context the model attends to.


In [9]:
def build_user_message(context, answer_text):
    return (
        f"Target Answer: {answer_text}\n"
        f"Generate a question from the following context where the target answer is the correct answer. "
        f"Do not include phrases like 'According to the text' in the question and do not repeat the context in the question.\n"
        f"Context: {context}"
    )


def preprocess(examples):
    all_input_ids, all_labels, all_attention_mask = [], [], []

    for context, answers, question in zip(examples["context"], examples["answers"], examples["question"]):
        answer_text = answers["text"][0] if len(answers["text"]) > 0 else ""
        user_msg = build_user_message(context, answer_text)

        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_msg},
        ]
        prompt_text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        full_text = prompt_text + question + tokenizer.eos_token

        prompt_ids = tokenizer(
            prompt_text, add_special_tokens=False, truncation=True, max_length=MAX_INPUT_LENGTH
        )["input_ids"]
        full_ids = tokenizer(
            full_text, add_special_tokens=False, truncation=True, max_length=MAX_SEQ_LENGTH
        )["input_ids"]

        labels = list(full_ids)
        prompt_len = min(len(prompt_ids), len(full_ids))
        for i in range(prompt_len):
            labels[i] = -100

        all_input_ids.append(full_ids)
        all_labels.append(labels)
        all_attention_mask.append([1] * len(full_ids))

    return {
        "input_ids": all_input_ids,
        "labels": all_labels,
        "attention_mask": all_attention_mask,
    }


tokenized_train = train_ds.map(preprocess, batched=True, remove_columns=train_ds.column_names)

# Dynamically pads input_ids/attention_mask/labels per batch (labels padded with -100).
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, label_pad_token_id=-100)


Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

## **Training arguments**

On Colab/Kaggle, set `bf16=True` if your GPU supports it (T4 only supports `fp16`; A100/L4/newer support `bf16`) - adjust the precision flags below if needed. Because TinyLlama (1.1B) is much smaller than Llama 3.2 3B, you generally have room to raise `per_device_train_batch_size` and/or lower `gradient_accumulation_steps` versus the 3B notebook if you want faster epochs; the values below are kept conservative so the notebook runs comfortably on a single T4. `paged_adamw_8bit` keeps the optimizer state memory-efficient, which matters most when `USE_4BIT=True`.


In [10]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    save_strategy="epoch",
    learning_rate=2e-4,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    weight_decay=0.01,
    num_train_epochs=TRAIN_EPOCH_SIZE,
    fp16=True,
    gradient_checkpointing=True,
    logging_steps=50,
    save_total_limit=2,
    optim="paged_adamw_8bit" if USE_4BIT else "adamw_torch",
    report_to="none",   # set to "wandb"/"tensorboard" if you use experiment tracking
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    processing_class=tokenizer,
    data_collator=data_collator,
)


## **Train the model**

In [11]:
trainer.train()


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss
50,1.510389
100,1.309116
150,1.315969
200,1.251242
250,1.274051
300,1.285429
350,1.245443
400,1.205324
450,1.186775
500,1.191441


TrainOutput(global_step=1250, training_loss=1.1113273071289063, metrics={'train_runtime': 6896.8696, 'train_samples_per_second': 2.9, 'train_steps_per_second': 0.181, 'total_flos': 5.47421127648215e+16, 'train_loss': 1.1113273071289063, 'epoch': 2.0})

## **Save the LoRA adapter**

Because this is a PEFT model, `save_model` / `save_pretrained` write out only the (small) LoRA adapter weights and config, not the full base model - the base model can always be re-downloaded from the Hub and combined with this adapter.


In [12]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Saved LoRA adapter to {OUTPUT_DIR}")


Saved LoRA adapter to /content/tinyllama-1.1b-qg-lora


## **Push the LoRA adapter to Huggingface Hub**

In [13]:
model.push_to_hub("gaurav-dey/tinyllama-1.1b-qg-lora")
tokenizer.push_to_hub("gaurav-dey/tinyllama-1.1b-qg-lora")


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

CommitInfo(commit_url='https://huggingface.co/gaurav-dey/tinyllama-1.1b-qg-lora/commit/a59df201eb40c6e9c728f7ddaf36b81420e0c249', commit_message='Upload tokenizer', commit_description='', oid='a59df201eb40c6e9c728f7ddaf36b81420e0c249', pr_url=None, repo_url=RepoUrl('https://huggingface.co/gaurav-dey/tinyllama-1.1b-qg-lora', endpoint='https://huggingface.co', repo_type='model', repo_id='gaurav-dey/tinyllama-1.1b-qg-lora'), pr_revision=None, pr_num=None)

## **Merge the model**

In [14]:
merged_model = model.merge_and_unload()


/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:397: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


## **Push the merged model to Huggingface Hub**

In [15]:
merged_model.push_to_hub("gaurav-dey/tinyllama-1.1b-qg")
tokenizer.push_to_hub("gaurav-dey/tinyllama-1.1b-qg")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

CommitInfo(commit_url='https://huggingface.co/gaurav-dey/tinyllama-1.1b-qg/commit/57215a55ce6829b1149448b232bcfb74cc33f616', commit_message='Upload tokenizer', commit_description='', oid='57215a55ce6829b1149448b232bcfb74cc33f616', pr_url=None, repo_url=RepoUrl('https://huggingface.co/gaurav-dey/tinyllama-1.1b-qg', endpoint='https://huggingface.co', repo_type='model', repo_id='gaurav-dey/tinyllama-1.1b-qg'), pr_revision=None, pr_num=None)

Optional: mount Google Drive and copy the checkpoint there so it persists after the Colab runtime disconnects.

## **Sanity-check generations**

In [16]:
model.eval()
tokenizer.padding_side = "left"   # left padding is what you want for batched generation

sample = val_ds.select(range(5))
for ex in sample:
    answer_text = ex["answers"]["text"][0] if ex["answers"]["text"] else ""
    user_msg = build_user_message(ex["context"], answer_text)

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_msg},
    ]
    prompt_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(
        prompt_text, return_tensors="pt", truncation=True, max_length=MAX_INPUT_LENGTH
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_TARGET_LENGTH,
            num_beams=4,
            do_sample=False,
        )

    generated_ids = output_ids[0][inputs["input_ids"].shape[1]:]
    generated_question = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

    print(f"Answer:              {answer_text}")
    print(f"Gold question:       {ex['question']}")
    print(f"Generated question:  {generated_question}")
    print("-" * 80)


[transformers] Both `max_new_tokens` (=96) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=96) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Answer:              1852
Gold question:       In what year did Massachusetts first require children to be educated in schools?
Generated question:  According to the context, what is the Supreme Court precedent regarding educational choice in the United States?
--------------------------------------------------------------------------------


[transformers] Both `max_new_tokens` (=96) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Answer:              1962
Gold question:       When were stromules discovered?
Generated question:  What is the correct answer to the question about stromules in chloroplasts? 

The correct answer to the question about stromules in chloroplasts is that they are rare in chloroplasts, and are much more common in other plastids like chromoplasts and amyloplasts in petals and roots, respectively.
--------------------------------------------------------------------------------


[transformers] Both `max_new_tokens` (=96) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Answer:              Horace Walpole
Gold question:       Which artist who had a major influence on the Gothic Revival is represented in the V&A's British galleries?
Generated question:  Who was a major influence on the Gothic Revival?
--------------------------------------------------------------------------------


[transformers] Both `max_new_tokens` (=96) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Answer:              several regional colleges and universities
Gold question:       In 1890, who did the university decide to team up with?
Generated question:  Which regional colleges and universities were affiliated with the University of Chicago in the 1890s?
--------------------------------------------------------------------------------
Answer:              Jonathan Stewart
Gold question:       Who got a touchdown making the score 10-7?
Generated question:  What was the correct answer to the question generated from the given context? 

The correct answer to the question generated from the given context is Jonathan Stewart, as he finished the Carolina Panthers' 9-play, 73-yard scoring drive with a 1-yard touchdown run, cutting the score to 10-7 with 11:28 left in the second quarter.
--------------------------------------------------------------------------------


## **Next steps**

- Set `USE_4BIT = False` to compare plain LoRA (16-bit base model) against QLoRA on quality and training speed - TinyLlama is small enough that plain LoRA is often the more practical default.
- Add the answer-highlighting (`<hl>` marker) preprocessing variant and compare against this baseline.
- Add a round-trip QA-consistency evaluation metric for a semantic-quality signal beyond ROUGE/BLEU.
- Since TinyLlama is much cheaper to train than the 3B model, consider raising `TRAIN_SUBSET_SIZE` toward the full SQuAD train split, or increasing `TRAIN_EPOCH_SIZE`, while keeping an eye on validation quality.
